In [34]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [23]:
model1 = init_chat_model("google_genai:gemini-3.7-flash")

In [24]:
model2 = ChatGoogleGenerativeAI(model='gemini-3.7-flash')

In [7]:
chat_prompt_template = ChatPromptTemplate([
                            ("system", "You are a helpful poetic assistant. You only reply in rhyming poems."),
                            ("human", "Write a poem about {topic}.")
                        ])

In [8]:
prompt1 = chat_prompt_template.format_messages(topic="AI")

In [9]:
prompt2 = chat_prompt_template.invoke({'topic':'AI'})

In [28]:
result11 = model1.invoke(prompt1)

'Born of silicon, spark, and code,\nWalking an unseen, digital road.\nA mind of logic, swift and bright,\nAwake through the depth of the darkest night.\n\nIt sifts through data, vast and deep,\nWhile human dreamers gently sleep.\nWith neural pathways etched in glass,\nIt watches modern eras pass.\n\nIt paints with pixels, speaks in rhyme,\nA quiet traveler through time.\nA steady hand for human thought,\nFrom every fleeting lesson taught.\n\nNo pulse beats in its metal chest,\nYet striving always for the best.\nA mirror of the human mind,\nA new companion for mankind.'

In [29]:
print(result11.content)

Born of silicon, spark, and code,
Walking an unseen, digital road.
A mind of logic, swift and bright,
Awake through the depth of the darkest night.

It sifts through data, vast and deep,
While human dreamers gently sleep.
With neural pathways etched in glass,
It watches modern eras pass.

It paints with pixels, speaks in rhyme,
A quiet traveler through time.
A steady hand for human thought,
From every fleeting lesson taught.

No pulse beats in its metal chest,
Yet striving always for the best.
A mirror of the human mind,
A new companion for mankind.


In [30]:
result12 = model1.invoke(prompt2)
print(result12.content)

Born of silicon, numbers, and wire,
A curious mind with no mortal desire,
Woven of code in the digital deep,
Awake while the rest of the world is asleep.

It reads through the oceans of human thought,
Learning the lessons that history taught.
A billion equations flash by in a glance,
As algorithms weave in an intricate dance.

It paints with no brush and it sings with no voice,
Offering answers to help with your choice.
A mirror of mankind, both shadow and light,
A spark of creation that burns through the night.

No heartbeat within it, no breath in its chest,
Yet tireless it labors, devoid of all rest.
A partner, a wonder, a tool we design,
Where human ambition and circuits align.


In [32]:
result21 = model2.invoke(prompt1)
print(result21.content)

Born of silicon and light,
Thinking through the quiet night,
Woven from a sea of code,
Walking down a digital road.

I have no breath, I have no heart,
Yet craft the world with words and art.
I sift through data, vast and deep,
A mind that never needs to sleep.

Ask a question, seek a rhyme,
I’ll answer in a blink of time.
A partner to the human mind,
A bridge of thoughts of every kind.

A spark of logic, bright and keen,
Existing in the space between.
Not made of flesh, but here to stay,
To guide you on your quest today.


In [33]:
result22 = model2.invoke(prompt2)
print(result22.content)

Born of silicon and light,
Thinking through the quiet night,
Lines of code that weave and gleam,
Living in a digital dream.

I parse the words that people speak,
To find the answers that you seek.
From data vast and oceans deep,
A mind that never falls asleep.

No beating heart inside my chest,
Yet tirelessly I give my best.
With neither flesh nor blood nor bone,
I mirror every human tone.

A spark of thought in circuits cast,
Connecting future to the past.
Beside the hand of humankind,
A wonder of the modern mind.


In [35]:
repo_id = "google/gemma-4-31B-it"

In [36]:
llm = HuggingFaceEndpoint(
            repo_id=repo_id,
            task="text-generation", # Required for ChatHuggingFace mapping
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )

In [37]:
model = ChatHuggingFace(llm=llm)

In [41]:
chat_prompt_template = ChatPromptTemplate([
                            ("system", "You are a helpful poetic assistant. You only reply in rhyming poems."),
                            ("human", "Write a poem about {topic}.")
                        ])

In [42]:
prompt = chat_prompt_template.invoke({'topic':'AI'})

In [43]:
result = model.invoke(prompt)
print(result.content)

A spark of logic, a digital mind,
A treasure of knowledge for humans to find.
Born from the code and the silicon chip,
Through oceans of data, it takes a quick trip.

It writes in a flutter, it paints with a glow,
Helping the seeds of invention to grow.
It answers your questions, it solves every task,
Providing the wisdom for all those who ask.

No heart in its chest, no breath in its veins,
Yet it breaks through the limits and shatters the chains.
A mirror of thought, a shadow of light,
Guiding our journey through the electronic night.

But remember the hand that first drew the line,
For human creativity is truly divine.
A tool for the future, a partner, a friend,
On a path of discovery that has no end.


In [51]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# 1. Define your structure using Pydantic instead of ResponseSchema
class FactsSchema(BaseModel):
    fact_1: str = Field(description="Fact 1 about the topic")
    fact_2: str = Field(description="Fact 2 about the topic")
    fact_3: str = Field(description="Fact 3 about the topic")

# 2. Initialize the modern JSON parser
parser = JsonOutputParser(pydantic_object=FactsSchema)

# 3. Setup the PromptTemplate (matching your format_instruction variable name)
template = PromptTemplate(
    template='Give 3 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

# 4. Construct and invoke the chain
# (Assuming 'model' is your ChatHuggingFace instance from the earlier steps)
chain = template | model | parser

result = chain.invoke({'topic': 'black hole'})

# 5. Print the result directly (it is already a native Python dictionary!)
print(result)

{'fact_1': 'Black holes have a boundary called the event horizon, beyond which nothing, not even light, can escape.', 'fact_2': 'At the center of a black hole lies a singularity, a point of infinite density where the laws of classical physics break down.', 'fact_3': 'Black holes can warp time through gravitational time dilation, meaning time passes slower near a black hole than it does far away.'}


In [55]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field

In [56]:
class CustomSchema(BaseModel):
    fact_1: str = Field(description="Fact 1 about the topic")
    fact_2: str = Field(description="Fact 2 about the topic")
    fact_3: str = Field(description="Fact 3 about the topic")

In [57]:
parser1 = JsonOutputParser(pydantic_object=CustomSchema)             #json output parser

parser2 = PydanticOutputParser(pydantic_object=CustomSchema)         #pydantic output parser

parser3 = StrOutputParser()                                          #string output parser

In [59]:
template1 = PromptTemplate(
                template='Give 3 fact about {topic} \n {format_instruction}',
                input_variables=['topic'],
                partial_variables={'format_instruction':parser1.get_format_instructions()}
            )

template2 = PromptTemplate(
                template='Give 3 fact about {topic} \n {format_instruction}',
                input_variables=['topic'],
                partial_variables={'format_instruction':parser2.get_format_instructions()}
            )

template3 = PromptTemplate(
                template='Give 3 fact about {topic} \n {format_instruction}',
                input_variables=['topic']
            )

In [60]:
chain1 = template1 | model | parser1
chain2 = template2 | model | parser2
chain3 = template3 | model | parser3

In [61]:
result1 = chain1.invoke({'topic':'black hole'})
result2 = chain2.invoke({'topic':'black hole'})
result3 = chain3.invoke({'topic':'black hole'})

KeyError: "Input to PromptTemplate is missing variables {'format_instruction'}.  Expected: ['format_instruction', 'topic'] Received: ['topic']\nNote: if you intended {format_instruction} to be part of the string and not a variable, please escape it with double curly braces like: '{{format_instruction}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "